# 365 Probabilidades - Dia #166
## Qual a probabilidade de a média de um grupo acertar mais do que você?

**Tipo:** Curiosidade · Decisão coletiva  
**Data de publicação:** 2026-11-26  
**Ferramenta:** Python  
**Decisão analisada:** Quando vale mais confiar no próprio palpite e quando vale juntar os palpites de muita gente?  
**Hashtag:** #365Probabilidades #Dia166

---

### 📖 A História

Em alguma festa, alguém já colocou um pote cheio de balas na mesa e pediu para cada convidado chutar quantas havia lá dentro.

Quase ninguém acerta. Mas dizem que, se alguém juntar todos os chutes, o palpite do grupo chega perto.

Fui ver se isso é verdade, e quando deixa de ser.

---

### 📚 O Conceito

A ideia tem nome, **sabedoria das multidões**, e uma origem precisa. Em 1907, o estatístico inglês Francis Galton publicou na revista *Nature* um artigo chamado "Vox Populi", a voz do povo.

A explicação é simples. Cada palpite é o valor verdadeiro mais um erro. Se os erros forem independentes, uns para cima e outros para baixo, eles se cancelam quando os palpites são juntados. Sobra algo perto do valor verdadeiro.

A condição é a parte difícil. Os erros precisam ser independentes. Quando todo mundo erra para o mesmo lado, juntar palpites não cancela nada.

---

### 🧮 O Modelo

Duas fontes e cinco contas.

1. **Os 787 palpites.** A distribuição dos palpites de Galton reconstruída a partir dos 19 percentis publicados, para calcular quantas pessoas chegaram tão perto quanto o palpite do grupo.
2. **O tamanho do grupo.** Grupos sorteados de 1 a 787 pessoas, para ver quantas são necessárias para o palpite do grupo ficar a 1% do peso real (200 mil sorteios no total, semente 42).
3. **Quando a multidão erra.** As oito perguntas de Navajas e colegas, com o erro do palpite do meio e da média da plateia em cada uma.
4. **O erro que não se cancela.** Uma simulação em que todos compartilham o mesmo erro, para mostrar o limite da sabedoria das multidões.
5. **Conversar antes.** O ganho de juntar decisões de pequenos grupos que discutiram, em vez de palpites isolados.

**Fontes:**
- Galton, F. (1907). Vox populi. *Nature*, 75, 450-451. **787 palpites** do peso de um boi numa feira em Plymouth, na Inglaterra. Lido no original, com a tabela de percentis.
- Wallis, K. F. (2014). Revisiting Francis Galton's forecasting competition. *Statistical Science*, 29(3), 420-424. Correção dos números de Galton a partir das anotações originais. Lido na versão do arXiv.
- Navajas, J., Niella, T., Garbulsky, G., Bahrami, B. & Sigman, M. (2018). Aggregated knowledge from a small number of debates outperforms the wisdom of large crowds. *Nature Human Behaviour*, 2(2), 126-132. **5.180 pessoas** numa plateia em Buenos Aires. Lido na versão do arXiv, com a tabela suplementar.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. As fontes são palpites registrados, e não autorrelato.

**Cordas declaradas:** #142 (o sobrenome que desaparece), que usou outro artigo de Galton.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE, N_SIM = 42, 200_000
rng = np.random.default_rng(SEMENTE)

def jeffreys(k, n, nivel=0.95):
    """Intervalo de Jeffreys para uma proporção."""
    a = (1 - nivel) / 2
    return stats.beta.ppf([a, 1 - a], k + 0.5, n - k + 0.5)

def sorteia(k, n):
    return rng.beta(k + 0.5, n - k + 0.5, N_SIM)

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# 1) Galton (1907), Nature 75, 450-451. Percentis dos 787 palpites do peso do boi abatido (libras).
percentis = np.arange(5, 100, 5)
palpites_pct = np.array([1074, 1109, 1126, 1148, 1162, 1174, 1181, 1188, 1197, 1207,
                         1214, 1219, 1225, 1230, 1236, 1243, 1254, 1267, 1293])
n_galton = 787
# Números publicados por Galton e corrigidos por Wallis (2014) a partir das anotações originais
publicado = dict(meio=1207, peso=1198)
corrigido = dict(meio=1208, peso=1197)

# 2) Navajas et al. (2018), Nature Human Behaviour 2(2). Tabela suplementar 1, respostas iniciais.
#    (pergunta, valor verdadeiro, média dos palpites, palpite do meio)
navajas = {
    'Gols na Copa de 2010':          (145, 107.9, 81),
    'Imperadores romanos':           (131, 19.5, 10),
    'Palavra numa letra de música':  (21, 26.7, 21),
    'Calorias em 200 g de manteiga': (1480, 1251.3, 600),
    'Soma dos números da roleta':    (666, 705.9, 400),
    'Barril de petróleo em 1970 (centavos)': (180, 6445.4, 150),
    'Altura da Torre Eiffel (m)':    (324, 344.4, 200),
    'Elevadores do Empire State':    (73, 18.6, 11),
}
navajas_n, navajas_grupos = 5180, 280
reducao = {'Plateia contra o indivíduo': 52.0,
           '4 consensos contra 1.400 palpites iniciais': 49.2,
           'Resposta revisada contra a inicial': 31.0}

aplica_fator_080 = False

fmt = lambda x, c=1: f"{x:.{c}f}".replace('.', ',')
print("=" * 72)
print("  DADOS DA LITERATURA - A SABEDORIA DAS MULTIDÕES")
print("=" * 72)
print(f"\n  Galton (1907): {n_galton} palpites do peso de um boi")
print(f"  → Publicado: palpite do meio {publicado['meio']} lb, peso {publicado['peso']} lb, erro {publicado['meio']-publicado['peso']} lb")
print(f"  → Corrigido (Wallis 2014): palpite do meio {corrigido['meio']} lb, peso {corrigido['peso']} lb, erro {corrigido['meio']-corrigido['peso']} lb")
print(f"  → Percentis 5 a 95: {', '.join(map(str, palpites_pct))}")
print(f"\n  Navajas et al. (2018): {navajas_n:,} pessoas".replace(',', '.'))
for q, (v, m, md) in navajas.items():
    print(f"     {q:<40} verdadeiro {v:>6}  ·  média {fmt(m):>7}  ·  meio {md:>5}")
for k, v in reducao.items():
    print(f"  → {k}: erro {fmt(v)}% menor")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")
print("=" * 72)

In [ ]:
# --- O MODELO ---
PESO = corrigido['peso']

# Conta 1 · Os 787 palpites, reconstruídos dos percentis (interpolação linear; caudas pela inclinação vizinha)
q = np.concatenate([[0], percentis, [100]]) / 100
v = np.concatenate([[palpites_pct[0] - (palpites_pct[1] - palpites_pct[0])], palpites_pct,
                    [palpites_pct[-1] + (palpites_pct[-1] - palpites_pct[-2])]])
posicoes = (np.arange(1, n_galton + 1) - 0.5) / n_galton
palpites = np.interp(posicoes, q, v)
meio = np.median(palpites)
erro_grupo = abs(meio - PESO)
erro_ind = np.abs(palpites - PESO)
p_pior = (erro_ind > erro_grupo).mean()
p_um_pct = (erro_ind <= 0.01 * PESO).mean()
erro_tipico = np.median(erro_ind)

# Conta 2 · O tamanho do grupo
tamanhos = [1, 3, 5, 10, 20, 50, 100, 200, 400, 787]
REP = 20_000
grupo = {}
for n in tamanhos:
    amostras = rng.choice(palpites, size=(REP, n), replace=True)
    meios = np.median(amostras, axis=1)
    sozinho = rng.choice(palpites, size=REP)
    grupo[n] = dict(dentro_1pct=(np.abs(meios - PESO) <= 0.01 * PESO).mean(),
                    vence_individuo=(np.abs(meios - PESO) < np.abs(sozinho - PESO)).mean())

# Conta 3 · Quando a multidão erra (Navajas)
erros_nav = {qq: dict(meio=(md - v_) / v_ * 100, media=(m - v_) / v_ * 100) for qq, (v_, m, md) in navajas.items()}
perto_meio = sum(abs(e['meio']) <= 10 for e in erros_nav.values())
perto_media = sum(abs(e['media']) <= 10 for e in erros_nav.values())

# Conta 4 · O erro que não se cancela (ilustração)
dp_ind = np.std(palpites)
vieses = [0.0, 0.03, 0.10]
erro_vs_n = {b: [] for b in vieses}
ns = [1, 5, 20, 100, 500, 2000]
for b in vieses:
    for n in ns:
        sim = PESO * (1 + b) + rng.normal(0, dp_ind, size=(2000, n))
        erro_vs_n[b].append(np.median(np.abs(sim.mean(axis=1) - PESO)) / PESO * 100)

print("=" * 72)
print("  MODELO - RESULTADO")
print("=" * 72)
print("\n  CONTA 1 · Os 787 palpites")
print(f"  → Palpite do meio reconstruído: {fmt(meio, 0)} lb  ·  erro do grupo: {fmt(erro_grupo, 0)} lb ({fmt(erro_grupo/PESO*100)}%)")
print(f"  → Erro típico de uma pessoa: {fmt(erro_tipico, 0)} lb ({fmt(erro_tipico/PESO*100)}%)")
print(f"  → P(uma pessoa errar mais que o grupo) = {fmt(p_pior*100)}%")
print(f"  → P(uma pessoa ficar a até 1% do peso) = {fmt(p_um_pct*100)}%")
print("\n  CONTA 2 · O tamanho do grupo")
for n, r in grupo.items():
    print(f"  → {n:>4} {'pessoa ' if n == 1 else 'pessoas'}: P(palpite do meio a até 1%) = {fmt(r['dentro_1pct']*100)}%  ·  P(vence uma pessoa sozinha) = {fmt(r['vence_individuo']*100)}%")
print("\n  CONTA 3 · Quando a multidão erra (Navajas, respostas iniciais)")
for qq, e in erros_nav.items():
    print(f"  → {qq:<40} palpite do meio {e['meio']:+.0f}%  ·  média {e['media']:+.0f}%".replace('.', ','))
print(f"  → Perguntas com erro de até 10%: {perto_meio} de 8 pelo palpite do meio, {perto_media} de 8 pela média")
print("\n  CONTA 4 · O erro que não se cancela (ilustração)")
for b in vieses:
    print(f"  → Viés comum de {fmt(b*100, 0)}%: erro do grupo com {ns[0]} a {ns[-1]} pessoas = " + ', '.join(fmt(x) for x in erro_vs_n[b]) + ' (%)')
print("\n  CONTA 5 · Conversar antes")
print(f"  → Quatro decisões de grupos de cinco tiveram erro {fmt(reducao['4 consensos contra 1.400 palpites iniciais'])}% menor que a média de 1.400 palpites isolados")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---
R1 = 'Fontes: Galton (1907), Nature 75; Wallis (2014). Conta própria | #365Probabilidades'

# GRÁFICO 1 · Os 787 palpites
fig1, ax1 = plt.subplots(figsize=(12, 8))
bins = np.arange(1020, 1340, 10)
ax1.hist(palpites, bins=bins, color=CINZA, alpha=0.55, edgecolor='white')
ax1.axvspan(PESO - erro_grupo, PESO + erro_grupo, color=DOURADO, alpha=0.3)
ax1.axvline(PESO, color='black', linewidth=2.5); ax1.axvline(meio, color=VERDE, linestyle='--', linewidth=2.5)
topo = ax1.get_ylim()[1]
ax1.text(PESO - 3, topo * 0.95, f'peso real\n{PESO} lb', ha='right', fontsize=12, fontweight='bold')
ax1.text(meio + 3, topo * 0.95, f'palpite do grupo\n{fmt(meio, 0)} lb', color=VERDE, fontsize=12, fontweight='bold')
ax1.text(1035, topo * 0.6, f"Só {fmt((1-p_pior)*100, 0)} em cada 100 pessoas\nchegaram tão perto\nquanto o palpite do grupo", fontsize=14, color='#8a6d1f', fontweight='bold')
ax1.set_xlabel('Palpite do peso do boi abatido (libras)'); ax1.set_ylabel('Pessoas')
ax1.set_title('A multidão errou por cerca de 1%. Quase todo mundo errou por mais\nOs 787 palpites de Plymouth, 1907, reconstruídos dos percentis', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, R1, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-166-grafico-01-os-787-palpites.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · O tamanho do grupo
fig2, ax2 = plt.subplots(figsize=(12, 8))
xs = list(grupo); y1 = [grupo[n]['dentro_1pct'] * 100 for n in xs]; y2 = [grupo[n]['vence_individuo'] * 100 for n in xs]
ax2.plot(xs, y1, 'o-', color=VERDE, linewidth=2.5, markersize=9, label='Palpite do grupo a até 1% do peso real')
ax2.plot(xs, y2, 's-', color=DOURADO, linewidth=2.5, markersize=9, label='Palpite do grupo melhor que o de uma pessoa sozinha')
for n in (10, 100):
    ax2.annotate(f"{fmt(grupo[n]['vence_individuo']*100, 0)}%", (n, grupo[n]['vence_individuo'] * 100), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=12, fontweight='bold')
    ax2.annotate(f"{fmt(grupo[n]['dentro_1pct']*100, 0)}%", (n, grupo[n]['dentro_1pct'] * 100), textcoords='offset points', xytext=(0, -20), ha='center', fontsize=12, fontweight='bold', color=VERDE)
ax2.set_xscale('log'); ax2.set_xticks(xs); ax2.set_xticklabels([str(n) for n in xs])
ax2.set_ylim(0, 105); ax2.set_xlabel('Pessoas no grupo'); ax2.set_ylabel('Chance (%)'); ax2.legend(frameon=False, loc='lower right')
ax2.set_title('Quanto mais gente, mais perto. Até certo ponto\nGrupos sorteados entre os 787 palpites de Galton', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, R1, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-166-grafico-02-tamanho-do-grupo.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Quando a multidão erra
fig3, ax3 = plt.subplots(figsize=(12, 8))
nomes = list(erros_nav); ys = np.arange(len(nomes))[::-1]
vals = [max(min(erros_nav[k]['meio'], 100), -100) for k in nomes]
cores = [VERDE if abs(x) <= 10 else (DOURADO if abs(x) <= 40 else VERMELHO) for x in vals]
ax3.barh(ys, vals, color=cores, height=0.6)
for y, x in zip(ys, vals):
    ax3.text(x + (2 if x >= 0 else -2), y, f"{x:+.0f}%", va='center', ha='left' if x >= 0 else 'right', fontsize=12, fontweight='bold')
ax3.axvline(0, color='black', linewidth=1.2)
ax3.set_yticks(ys); ax3.set_yticklabels(nomes, fontsize=12); ax3.set_xlim(-115, 40)
ax3.set_xlabel('Erro do palpite do meio da plateia em relação ao valor verdadeiro (%)')
ax3.set_title('A mesma multidão acertou uma pergunta e errou feio outras\n5.180 pessoas, oito perguntas, Buenos Aires, 2015', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, 'Fonte: Navajas, Niella, Garbulsky, Bahrami & Sigman (2018), Nature Human Behaviour 2(2). Conta própria | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-166-grafico-03-quando-a-multidao-erra.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · O erro que não se cancela
fig4, ax4 = plt.subplots(figsize=(12, 8))
cores4 = {0.0: VERDE, 0.03: DOURADO, 0.10: VERMELHO}
for b in vieses:
    ax4.plot(ns, erro_vs_n[b], 'o-', color=cores4[b], linewidth=2.5, markersize=9,
             label='Erros independentes' if b == 0 else f'Todos erram {fmt(b*100, 0)}% para o mesmo lado')
ax4.set_xscale('log'); ax4.set_xticks(ns); ax4.set_xticklabels([str(n) for n in ns])
ax4.set_xlabel('Pessoas no grupo'); ax4.set_ylabel('Erro típico do palpite do grupo (%)'); ax4.legend(frameon=False, loc='center right')
ax4.set_title('Juntar palpites cancela os erros que vão para lados diferentes, e não o erro de todos\nSimulação ilustrativa com a dispersão dos palpites de Galton', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, 'Simulação própria a partir de Galton (1907) | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-166-grafico-04-o-erro-que-fica.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Em 1907, o palpite do meio de 787 pessoas errou o peso do boi por cerca de 1%. Só 11 em cada 100 pessoas chegaram tão perto sozinhas.**

A média de um grupo acerta mais do que você em quase 9 de cada 10 vezes, quando os erros vão para lados diferentes. Com 10 pessoas, o palpite do grupo já supera o de uma pessoa sozinha em 77% das vezes.

Mas a mesma multidão que acerta uma pergunta erra feio outra. Em 2015, 5.180 pessoas em Buenos Aires acertaram em cheio quantas vezes uma palavra aparecia numa música. E chutaram, no palpite do meio, 10 imperadores romanos, quando foram 131. Quando quase todo mundo erra para o mesmo lado, juntar mais gente não ajuda.

O que melhorou a multidão foi conversar. Quatro grupos de cinco pessoas que discutiram antes de responder tiveram erro 49% menor do que a média de 1.400 palpites isolados.

A pergunta que fica:

*Na última decisão importante que você tomou sozinho, de quem era o palpite que faltou ouvir?*

---

### ⚠️ Limitações do Modelo

- Os 787 palpites de Galton não foram publicados um a um. Eles foram reconstruídos a partir dos 19 percentis da tabela, por interpolação linear. As caudas abaixo do 5º e acima do 95º percentil foram estendidas pela inclinação vizinha, e por isso as contas que dependem dos extremos são aproximadas.
- A tabela foi transcrita de uma digitalização do original e conferida nos quartis e nos extremos citados por Wallis (2014). Pode haver erro de transcrição em algum percentil intermediário.
- Galton publicou o palpite do meio como 1.207 libras e o peso como 1.198. Wallis (2014) mostrou, pelas anotações originais, que os valores corretos são 1.208 e 1.197. As contas usam o peso de 1.197. Galton usou o palpite do meio, e não a média, e informou a média só depois, em carta, a pedido de um leitor.
- Na Conta 2, os grupos são sorteados com reposição entre os mesmos 787 palpites. Por isso, mesmo o grupo de 787 pessoas não acerta sempre a 1%.
- A Conta 4 é uma ilustração, com vieses escolhidos por nós (3% e 10%), e não uma medida de nenhum estudo.
- Navajas e colegas medem o erro de forma padronizada, e os percentuais por pergunta da Conta 3 são contas nossas sobre a tabela suplementar. A plateia era de um evento de palestras em Buenos Aires, e as perguntas eram de conhecimento geral, em que a resposta certa existe e pode ser conferida.
- As análises de grupo de Navajas usam 280 grupos completos, e não os 5.180 participantes.
- A pergunta do título fala em "média de um grupo". Galton usou o palpite do meio, que é a mediana. Nas perguntas de Navajas, a média e a mediana às vezes contam histórias diferentes. No preço do barril de petróleo, a mediana errou 17%, e a média errou mais de 3.000%, por causa de poucos palpites absurdos.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*